In [1]:
import os

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write("""{
        "file_format_version" : "1.0.0",
        "ICD" : {
            "library_path" : "libEGL_nvidia.so.0"
        }
    }
    """)
        
os.environ["MUJOCO_GL"] = "egl"

import gymnasium
import numpy as np
import torch
import yaml
from PIL import Image
from torchvision.transforms import v2
from einops import rearrange

from config import METAWORLD_CFGS, DMC_CFGS
from metaworld_env import setup_metaworld_env
from dmc import setup_dmc_env

from model.encoder import EncoderNet, FrameObservationEncoderNet
from model.actor import DDPGActorNet
from model.critic import CriticNet

In [2]:
task_name = "pickplace"
seed = 0

In [3]:
def setup_env(task_name:str, seed:int, render_mode:str|None = None) -> gymnasium.Env:
    if task_name in METAWORLD_CFGS:
        env = setup_metaworld_env(task_name, seed, render_mode)
    elif task_name in DMC_CFGS:
        env = setup_dmc_env(task_name, seed, render_mode)
    env = gymnasium.wrappers.RecordEpisodeStatistics(env)
    env = gymnasium.wrappers.AddRenderObservation(env, render_only=True)
    env = gymnasium.wrappers.FrameStackObservation(env, 2)

    return env

In [4]:
if task_name in METAWORLD_CFGS:
    config_path = "configs/ddpg_metaworld.yaml"
elif task_name in DMC_CFGS:
    config_path = "configs/ddpg_dmc.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
envs = setup_env(task_name, 5000, "rgb_array")

obs_dim = envs.observation_space.shape
action_dim = envs.action_space.shape

encoder = EncoderNet(39, config["encoder_layers"]).to(device)
actor = DDPGActorNet(encoder.dim, np.prod(action_dim), config["actor_layers"]).to(device)
critic = CriticNet(encoder.dim, np.prod(action_dim), config["critic_layers"]).to(device)

frame_encoder = FrameObservationEncoderNet(6, encoder.dim).to(device)

encoder_weight, actor_weight, critic_weight = torch.load(f"weights/ddpg/{task_name}/actor_{seed}_100.pt", weights_only=True)
encoder.load_state_dict(encoder_weight)
actor.load_state_dict(actor_weight)
critic.load_state_dict(critic_weight)

frame_encoder_weight = torch.load(f"weights/ddpg/{task_name}/frame_encoder_{seed}_mse.pt", weights_only=True)
frame_encoder.load_state_dict(frame_encoder_weight)
encoder = frame_encoder

encoder = encoder.eval()
actor = actor.eval()

transform = v2.Compose([
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.485, 0.456, 0.406, 0.485, 0.456, 0.406), (0.229, 0.224, 0.225, 0.229, 0.224, 0.225)),
        ])

In [ ]:
def preprpcess(obs_batch):
    obs_batch = rearrange(obs_batch, "b l h w c -> b (l c) h w")
    obs_batch = transform(obs_batch)
    return obs_batch

@torch.no_grad()
def get_action(obs_batch, deterministic, random):

    obs_batch = torch.as_tensor(obs_batch, dtype=torch.float32).unsqueeze(0).to(device)
    #obs_batch = torch.as_tensor(obs_batch).unsqueeze(0).to(device)
    obs_batch = preprpcess(obs_batch)
    obs_batch = encoder(obs_batch)

    obs_batch = torch.nn.functional.silu(obs_batch)
    dist = actor(obs_batch, 1)
    if deterministic:
        action = dist.mean
    else:    
        action = dist.sample(clip=None)

        if random:
            action.uniform_(-1, 1)
    
    action = action.cpu().numpy()
    
    return action.tolist()

In [30]:
def rollout():
    obs, info = envs.reset()
    success = 0
    for i in range(500):
        action = get_action(obs, deterministic=True, random=False)
        next_obs, reward, done, timeout, info = envs.step(action[0])
        success += info["success"]
        obs = next_obs

    print(success)

In [31]:
obs, info = envs.reset()
success = 0
imgs = []
for i in range(500):
    imgs.append(Image.fromarray(envs.render()))
    action = get_action(obs, False, False)
    next_obs, reward, done, timeout, info = envs.step(action[0])
    #success += info["success"]
    obs = next_obs

imgs[0].save('output.gif',
             save_all=True,
             append_images=imgs[1:],  # Remaining frames
             duration=100,              # Duration per frame in ms
             loop=0)

In [32]:
info

{'success': 0.0,
 'near_object': 0.0,
 'grasp_success': 0.0,
 'grasp_reward': 0.0029688553494315218,
 'in_place_reward': 0.13977244022745916,
 'obj_to_target': 0.28645739709662554,
 'unscaled_reward': 0.0029155824697524994,
 'motion': 'unwanted action for push back.',
 'is_grasped': False,
 'episode': {'r': 2.3813365401065996, 'l': 500, 't': 2.19907}}